In [1]:
!pip install git+https://huggingface.co/kontur-ai/sbert_punc_case_ru -q
!pip install pandas openpyxl tqdm -q

!pip install -q accelerate==1.3.0
!pip install -q "transformers>=5.0.0"
!pip install -U bitsandbytes>=0.46.1
!pip install pandas openpyxl tqdm -q

!pip install gigachat -q

from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

import os
import time
from typing import List, Optional, Tuple

import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from sbert_punc_case_ru import SbertPuncCase

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.8 MB/s eta 0:00:00


In [42]:
SEG_FILE = 'second_dataset.csv' # НАЗВАНИЕ ДАТАСЕТА

df_seg = pd.read_csv(SEG_FILE)
segments = df_seg['segment'].dropna().tolist()

segments = segments[:200]  # берём первые 200 сегментов

In [43]:
class BasePunctuator:

    def process(self, segment: str) -> Optional[str]:
        raise NotImplementedError

    def flush(self) -> Optional[str]:
        # обработка концовки, которая короче буфера
        raise NotImplementedError

def process_segments(punctuator: BasePunctuator, segments: List[str],
                     show_progress: bool = True) -> Tuple[List[str], List[float], List[int]]:
    # прогон сегментов через пунктуатор

    chunks = []
    times = []
    sizes = []

    buffer_start_time = None
    buffer_filled = 0

    iterator = tqdm(segments, desc="Обработка", unit="сегм", ncols=90) if show_progress else segments

    for seg in iterator:
        if buffer_filled == 0:
            buffer_start_time = time.time()

        res = punctuator.process(seg)
        buffer_filled += 1

        if res:
            chunks.append(res)
            times.append(time.time() - buffer_start_time)
            sizes.append(buffer_filled)
            buffer_filled = 0
            buffer_start_time = None

    if show_progress:
        iterator.set_description("Flush буфера")
    final = punctuator.flush()
    if final:
        chunks.append(final)
        if buffer_start_time is not None:
            times.append(time.time() - buffer_start_time)
        sizes.append(buffer_filled)
        buffer_filled = 0

    if show_progress:
        iterator.close()

    return chunks, times, sizes

def save_one_prototype(name: str, segments: List[str],
                       chunks: List[str], times: List[float], sizes: List[int],
                       filename: str = 'punctuation_second_dataset_200.xlsx'): # МЕНЯЕМ НАЗВАНИЕ ФАЙЛА ДЛЯ СОХРАНЕНИЯ

    if os.path.exists(filename):
        df = pd.read_excel(filename)
    else:
        df = pd.DataFrame()
        df['segment_original'] = segments

    result_col = []
    delay_col = []
    for chunk, delay, size in zip(chunks, times, sizes):
        result_col.extend([chunk] * size)
        delay_col.extend([delay] * size)

    df[f'{name}_result'] = result_col
    df[f'{name}_delay_sec'] = delay_col

    df.to_excel(filename, index=False)

In [12]:
class SbertBufferPunctuator(BasePunctuator):

    def __init__(self, buffer_size: int = 1):

        self.model = SbertPuncCase()
        self.buffer_size = buffer_size
        self.buffer = [] # список сегментов, попавших в буфер на текущем шаге

    def _process_buffer(self, segments: List[str]) -> str:

        if not segments:
            return ''
        full_text = ' '.join(segments)
        return self.model.punctuate(full_text)

    def process(self, segment: str) -> Optional[str]:

        self.buffer.append(segment)
        if len(self.buffer) >= self.buffer_size:
            buff = self.buffer.copy()
            self.buffer.clear()
            return self._process_buffer(buff)
        return None

    def flush(self) -> Optional[str]:

        if self.buffer:
            result = self._process_buffer(self.buffer)
            self.buffer.clear()
            return result
        return None

In [44]:
NAME = 'SbertBuffer'
punct = SbertBufferPunctuator(buffer_size=1)
chunks, times, sizes = process_segments(punct, segments)
save_one_prototype(NAME, segments, chunks, times, sizes)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Обработка: 100%|██████████████████████████████████████| 200/200 [01:33<00:00,  2.15сегм/s]


In [14]:
class SbertWindowPunctuator(BasePunctuator):
    """
    SbertPuncCase учитывает до 5 предыдущих сегментов.
    """

    def __init__(self, context_size: int = 5):
        self.model = SbertPuncCase()
        self.context_size = context_size
        self.buffer = []

    def _punctuate_full(self, segments: List[str]) -> str:
        if not segments:
            return ''
        return self.model.punctuate(' '.join(segments))

    def process(self, segment: str) -> Optional[str]:
        self.buffer.append(segment)
        if len(self.buffer) > self.context_size:
            self.buffer = self.buffer[-self.context_size:]

        formatted = self._punctuate_full(self.buffer)

        n = len(segment.split())
        return ' '.join(formatted.split()[-n:])

    def flush(self) -> Optional[str]:
        return None

In [45]:
NAME = 'SbertWindow5'
punct = SbertWindowPunctuator(context_size=5)
chunks, times, sizes = process_segments(punct, segments)
save_one_prototype(NAME, segments, chunks, times, sizes)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Обработка: 100%|██████████████████████████████████████| 200/200 [05:11<00:00,  1.56s/сегм]


In [50]:
class SpellCorrectorPunctuator(BasePunctuator):
    """
    Пунктуатор на основе melsmm/Spell-Corrector-RU-4B.
    Буфер size=1: каждый сегмент обрабатывается отдельно.
    Загружается в 4-битном квантовании для экономии VRAM.
    """

    def __init__(self, buffer_size: int = 1):
        self.buffer_size = buffer_size
        self.buffer = []
        self.model_name = "melsmm/Spell-Corrector-RU-4B"
        self._load_model()

    def _load_model(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.float16,
        )

    def _process_text(self, text: str) -> str:
        if not text or not text.strip():
            return text

        # prompt = (
        #     f"Исходный текст (фрагмент потока распознавания речи, без пунктуации):\n"
        #     f"{text}\n\n"
        #     f"Отредактируй исходный текст, расставив знаки препинания. "
        #     f"Учти, что это фрагмент: предложение может продолжаться в следующем фрагменте, "
        #     f"поэтому в конце допустима запятая или отсутствие знака."
        #     )

        prompt = (
            f"Отредактируй исходный текст, расставив знаки препинания."
            f"{text}\n\n"
            )


        messages = [{"role": "user", "content": prompt}]

        encoded = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to(self.model.device)

        input_ids = encoded["input_ids"]
        attention_mask = encoded.get("attention_mask")

        with torch.no_grad():
            outputs = self.model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=256,
                temperature=0.1,
                top_p=0.7,
                do_sample=True,
                )

        generated = outputs[0][input_ids.shape[1]:]
        result = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
        return result

    def _process_buffer(self, segments: List[str]) -> str:
        if not segments:
            return ''
        full_text = ' '.join(segments)
        return self._process_text(full_text)

    def process(self, segment: str) -> Optional[str]:
        self.buffer.append(segment)
        if len(self.buffer) >= self.buffer_size:
            buff = self.buffer.copy()
            self.buffer.clear()
            return self._process_buffer(buff)
        return None

    def flush(self) -> Optional[str]:
        if self.buffer:
            result = self._process_buffer(self.buffer)
            self.buffer.clear()
            return result
        return None

In [51]:
NAME = 'SpellCorrector_simple_prompt'
punct = SpellCorrectorPunctuator(buffer_size=1)
chunks, times, sizes = process_segments(punct, segments)
save_one_prototype(NAME, segments, chunks, times, sizes)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Обработка: 100%|██████████████████████████████████████| 200/200 [04:09<00:00,  1.25s/сегм]


In [53]:
class SpellCorrectorWindowPunctuator(BasePunctuator):

    def __init__(self, model, tokenizer, context_size: int = 5):
        self.model = model
        self.tokenizer = tokenizer
        self.context_size = context_size
        self.buffer = []

    def _process_text(self, text: str) -> str:
        if not text or not text.strip():
            return text

        prompt = (
            f"Исходный текст (фрагмент потока распознавания речи, без пунктуации):\n"
            f"{text}\n\n"
            f"Отредактируй исходный текст, расставив знаки препинания. "
            f"Учти, что это фрагмент: предложение может продолжаться в следующем фрагменте, "
            f"поэтому в конце допустима запятая или отсутствие знака."
        )

        # prompt = (
        #     f"Отредактируй исходный текст, расставив знаки препинания."
        #     )
        #     f"{text}\n\n"

        messages = [{"role": "user", "content": prompt}]

        encoded = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        ).to(self.model.device)

        input_ids = encoded["input_ids"]
        attention_mask = encoded.get("attention_mask")

        with torch.no_grad():
            outputs = self.model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=256,
                temperature=0.1,
                top_p=0.7,
                do_sample=True,
            )

        generated = outputs[0][input_ids.shape[1]:]
        result = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
        return result

    def process(self, segment: str) -> Optional[str]:
        self.buffer.append(segment)
        if len(self.buffer) > self.context_size:
            self.buffer = self.buffer[-self.context_size:]

        formatted = self._process_text(' '.join(self.buffer))

        # Возвращаем только последний сегмент
        n = len(segment.split())
        return ' '.join(formatted.split()[-n:])

    def flush(self) -> Optional[str]:
        return None

In [54]:
NAME = 'SpellCorrectorWindow5'
punct_window = SpellCorrectorWindowPunctuator(
    model=punct.model,
    tokenizer=punct.tokenizer,
    context_size=5,
)
chunks, times, sizes = process_segments(punct_window, segments)
save_one_prototype(NAME, segments, chunks, times, sizes)

Обработка: 100%|██████████████████████████████████████| 200/200 [09:07<00:00,  2.74s/сегм]


In [55]:
class GigaChatPunctuator(BasePunctuator):
    """
    Пунктуатор на основе GigaChat API.
    Буфер size=1: каждый сегмент обрабатывается отдельно.
    """

    def __init__(self, buffer_size: int = 1, credentials: str = "",
                 model: str = "GigaChat-3-Ultra"):
        self.buffer_size = buffer_size
        self.buffer = []
        self.model = model
        self.client = GigaChat(
            base_url="https://api.giga.chat/v1",
            credentials=credentials,
            scope="GIGACHAT_API_PERS",
            verify_ssl_certs=False,
        )

    def _process_text(self, text: str) -> str:
        if not text or not text.strip():
            return text

        prompt = (
            f"Исходный текст (фрагмент потока распознавания речи, без пунктуации):\n"
            f"{text}\n\n"
            f"Отредактируй исходный текст, расставив знаки препинания. "
            f"Учти, что это фрагмент: предложение может продолжаться в следующем фрагменте, "
            f"поэтому в конце допустима запятая или отсутствие знака."
            f"Возвращай ТОЛЬКО отредактированный фрагмент, без лишних символов, пробелов и пояснений."
        )

        chat = Chat(
            model=self.model,
            messages=[Messages(role=MessagesRole.USER, content=prompt)],
        )

        resp = self.client.chat(chat)
        return resp.choices[0].message.content.strip()

    def _process_buffer(self, segments: List[str]) -> str:
        if not segments:
            return ''
        return self._process_text(' '.join(segments))

    def process(self, segment: str) -> Optional[str]:
        self.buffer.append(segment)
        if len(self.buffer) >= self.buffer_size:
            buff = self.buffer.copy()
            self.buffer.clear()
            return self._process_buffer(buff)
        return None

    def flush(self) -> Optional[str]:
        if self.buffer:
            result = self._process_buffer(self.buffer)
            self.buffer.clear()
            return result
        return None

In [56]:
GIGACHAT_CREDENTIALS = "" # СЮДА КЛЮЧ

NAME = 'GigaChatUltra'
punct_giga = GigaChatPunctuator(
    buffer_size=1,
    credentials=GIGACHAT_CREDENTIALS,
    model="GigaChat-3-Ultra",
)
chunks, times, sizes = process_segments(punct_giga, segments)
save_one_prototype(NAME, segments, chunks, times, sizes)

Обработка: 100%|██████████████████████████████████████| 200/200 [01:53<00:00,  1.77сегм/s]


In [29]:
class GigaChatWindowPunctuator(BasePunctuator):

    def __init__(self, context_size: int = 5, credentials: str = "",
                 model: str = "GigaChat-3-Ultra"):
        self.context_size = context_size
        self.buffer = []
        self.model = model
        self.client = GigaChat(
            base_url="https://api.giga.chat/v1",
            credentials=credentials,
            scope="GIGACHAT_API_PERS",
            verify_ssl_certs=False,
        )

    def _process_text(self, text: str) -> str:
        if not text or not text.strip():
            return text

        prompt = (
            f"Исходный текст (фрагмент потока распознавания речи, без пунктуации):\n"
            f"{text}\n\n"
            f"Отредактируй исходный текст, расставив знаки препинания. "
            f"Учти, что это фрагмент: предложение может продолжаться в следующем фрагменте, "
            f"поэтому в конце допустима запятая или отсутствие знака."
            f"Возвращай ТОЛЬКО отредактированный фрагмент, без лишних символов, пробелов и пояснений."
        )

        chat = Chat(
            model=self.model,
            messages=[Messages(role=MessagesRole.USER, content=prompt)],
        )

        resp = self.client.chat(chat)
        return resp.choices[0].message.content.strip()

    def process(self, segment: str) -> Optional[str]:
        # Добавляем новый сегмент в буфер
        self.buffer.append(segment)

        # Обрезаем буфер до последних context_size сегментов
        if len(self.buffer) > self.context_size:
            self.buffer = self.buffer[-self.context_size:]

        # Прогоняем весь буфер через модель
        formatted = self._process_text(' '.join(self.buffer))

        # Возвращаем только последний сегмент (по количеству слов)
        n = len(segment.split())
        return ' '.join(formatted.split()[-n:])

    def flush(self) -> Optional[str]:
        return None

In [57]:
GIGACHAT_CREDENTIALS = "" # СЮДА КЛЮЧ

NAME = 'GigaChatWindow5'
punct_giga_window = GigaChatWindowPunctuator(
    context_size=5,
    credentials=GIGACHAT_CREDENTIALS,
    model="GigaChat-3-Ultra",
)
chunks, times, sizes = process_segments(punct_giga_window, segments)
save_one_prototype(NAME, segments, chunks, times, sizes)

Обработка: 100%|██████████████████████████████████████| 200/200 [02:48<00:00,  1.19сегм/s]


SAGE

https://huggingface.co/ai-forever/sage-v1.1.0

In [ ]:
class SagePunctuator(BasePunctuator):
    """
    Буфер size=1: каждый сегмент обрабатывается отдельно.
    """

    def __init__(self, buffer_size: int = 1):
        self.buffer_size = buffer_size
        self.buffer = []
        self.tokenizer_name = "ai-forever/FRED-T5-1.7B"
        self.model_name = "ai-forever/sage-v1.1.0"
        self._load_model()

    def _load_model(self):
        from transformers import AutoTokenizer, T5ForConditionalGeneration
        self.tokenizer = AutoTokenizer.from_pretrained(self.tokenizer_name)
        self.model = T5ForConditionalGeneration.from_pretrained(
            self.model_name,
            torch_dtype=torch.float16,
        )
        self.model.to('cuda')
        self.model.eval()

    def _process_text(self, text: str) -> str:
        if not text or not text.strip():
            return text

        input_text = "<LM>" + text # префикс, который ожидает модель
        encodings = self.tokenizer(
            input_text,
            max_length=None,
            padding='longest',
            truncation=False,
            return_tensors="pt",
        )
        for k, v in encodings.items():
            encodings[k] = v.to('cuda:0')

        with torch.inference_mode():
            res = self.model.generate(
                **encodings,
                use_cache=True,
                max_length=int(encodings['input_ids'].size(1) * 1.5),
            )
        res = res.cpu().tolist()
        result = self.tokenizer.batch_decode(res, skip_special_tokens=True)[0]
        return result.strip()

    def _process_buffer(self, segments: List[str]) -> str:
        if not segments:
            return ''
        full_text = ' '.join(segments)
        return self._process_text(full_text)

    def process(self, segment: str) -> Optional[str]:
        self.buffer.append(segment)
        if len(self.buffer) >= self.buffer_size:
            buff = self.buffer.copy()
            self.buffer.clear()
            return self._process_buffer(buff)
        return None

    def flush(self) -> Optional[str]:
        if self.buffer:
            result = self._process_buffer(self.buffer)
            self.buffer.clear()
            return result
        return None

In [ ]:
NAME = 'SAGE'
punct = SagePunctuator(buffer_size=1)
chunks, times, sizes = process_segments(punct, segments)
save_one_prototype(NAME, segments, chunks, times, sizes)

In [ ]:
class SageWindowPunctuator(BasePunctuator):
    """
    SAGE с левым контекстом
    Переиспользует уже загруженную модель из существующего объекта, поэтому надо загружать сначала прошлый прототип.
    """

    def __init__(self, model, tokenizer, context_size: int = 5):
        self.model = model
        self.tokenizer = tokenizer
        self.context_size = context_size
        self.buffer = []

    def _process_text(self, text: str) -> str:
        if not text or not text.strip():
            return text

        input_text = "<LM>" + text
        encodings = self.tokenizer(
            input_text,
            max_length=None,
            padding='longest',
            truncation=False,
            return_tensors="pt",
        )
        for k, v in encodings.items():
            encodings[k] = v.to('cuda:0')

        with torch.inference_mode():
            res = self.model.generate(
                **encodings,
                use_cache=True,
                max_length=int(encodings['input_ids'].size(1) * 1.5),
            )
        res = res.cpu().tolist()
        result = self.tokenizer.batch_decode(res, skip_special_tokens=True)[0]
        return result.strip()

    def process(self, segment: str) -> Optional[str]:
        self.buffer.append(segment)
        if len(self.buffer) > self.context_size:
            self.buffer = self.buffer[-self.context_size:]
        formatted = self._process_text(' '.join(self.buffer))
        n = len(segment.split())
        return ' '.join(formatted.split()[-n:])

    def flush(self) -> Optional[str]:
        return None

In [ ]:
NAME = 'SAGEWindow5'
punct_window = SageWindowPunctuator(
    model=punct.model,
    tokenizer=punct.tokenizer,
    context_size=5,
)
chunks, times, sizes = process_segments(punct_window, segments)
save_one_prototype(NAME, segments, chunks, times, sizes)